# Eight-step CatMAP + C/O/H-family BEP screening + activity volcano

This is a parallel copy of the original `lyq_alkene_epoxidation` framework. All eight reactions and original formation energies remain intact. Excel NEB three-state data is used **only** to screen adsorption-energy descriptors against a provisional barrier, not silently substituted for a different elementary TS.


In [ ]:
# BEP-like descriptor screening from original species energies and workbook NEB data.
from pathlib import Path
import os
import runpy
import numpy as np

here = Path.cwd()
if not (here / "alkene_epoxidation.mkm").is_file():
    choices = [here / "lyq_alkene_epoxidation1" / "2-creating_microkinetic_model",
               here / "2-creating_microkinetic_model"]
    found = [x for x in choices if (x / "alkene_epoxidation.mkm").exists()]
    if not found:
        raise FileNotFoundError("Open the lyq_alkene_epoxidation1 project notebook folder.")
    here = found[0]
os.chdir(here)
analysis = runpy.run_path(str(Path("../1-generating_input_file/bep_descriptor_analysis.py").resolve()))
bep_rows, bep_best = analysis["run"](make_plots=True)
print("Best screened pair:", bep_best["descriptors"])
print("Selected CatMAP descriptor axes: O_s, Ha_s")


In [ ]:
# Original eight elementary reactions preserved; only the screened descriptor axes differ.
from catmap import ReactionModel
from catmap import analyze
import matplotlib.pyplot as plt

mkm_file = "alkene_epoxidation.mkm"
model = ReactionModel(setup_file=mkm_file)
for variable in ["production_rate", "rate", "coverage"]:
    if variable not in model.output_variables:
        model.output_variables.append(variable)
model.force_recalculation = True
for attr in ["adsorbate_coefficient_matrix", "coefficient_matrix",
             "total_coefficient_dict", "transition_state_coefficient_matrix",
             "transition_state_scaling_matrix", "kBT",
             "coverage_map", "rate_map", "production_rate_map"]:
    if attr not in model._pickle_attrs:
        model._pickle_attrs.append(attr)
assert tuple(model.descriptor_names) == ("O_s", "Ha_s")
assert len(model.rxn_expressions) == 8
model.run()
resolution = model.resolution
expected = int(np.prod(resolution)) if isinstance(resolution,(list,tuple)) else int(resolution)**2
print("Descriptor grid:", model.descriptor_names)
print("Reaction steps:", len(model.rxn_expressions))
for variable in ["production_rate", "rate", "coverage"]:
    points = len(getattr(model, variable+"_map"))
    print(variable, "points:",points)
    if points != expected:
        raise RuntimeError(f"Incomplete {variable} map: {points}/{expected}")


In [ ]:
# Original-rate, all-rates, product-rate, coverage, and scaling figures.
# Fallback path avoids Windows PermissionError if an old PDF is still open.
from datetime import datetime

backup_dir = Path("plots")/("rerun_"+datetime.now().strftime("%Y%m%d_%H%M%S"))

def save_plot(plotter,name):
    try:
        fig=plotter.plot(save=name)
        p=Path(name)
    except PermissionError:
        backup_dir.mkdir(parents=True,exist_ok=True)
        p=backup_dir/name
        print("WARNING: locked existing PDF; fallback:",p)
        fig=plotter.plot(save=str(p))
    if fig is not None:
        plt.close(fig)
    print("Saved:",p)
    return p

vm=analyze.VectorMap(model)
vm.plot_variable="rate"
vm.log_scale=False  # signed net elementary rates; no log10 of negative rates
vm.unique_only=True
save_plot(vm,"rate.pdf")
vm.unique_only=False
save_plot(vm,"all_rates.pdf")

vm=analyze.VectorMap(model)
vm.plot_variable="production_rate"
vm.include_labels=["C6H12O_g"]
prod_labels=list(model.output_labels["production_rate"])
idx=prod_labels.index("C6H12O_g")
vals=np.asarray([float(v[idx]) for pt,v in model.production_rate_map])
vm.log_scale=bool(np.all(vals>0)) # avoid log(0) and log(negative)
vm.threshold=1e-30 if vm.log_scale else 0
if vm.log_scale:
    vm.min=1e-30
vm.descriptor_labels=["O* formation energy (eV)","Ha* formation energy (eV)"]
save_plot(vm,"production_rate.pdf")
save_plot(vm,"pretty_production_rate.pdf")

vm=analyze.VectorMap(model)
vm.plot_variable="coverage"
vm.log_scale=False
vm.min,vm.max=0,1
save_plot(vm,"coverage.pdf")
vm.include_labels=["O_s","Ha_s"]
save_plot(vm,"O_Ha_coverage.pdf")

sa=analyze.ScalingAnalysis(model)
save_plot(sa,"scaling.pdf")


In [ ]:
# True two-descriptor CatMAP activity map / volcano-like contour.
# Unlike the BEP plane, color = CatMAP-computed epoxide production rate.
import csv
import matplotlib.pyplot as plt

pts=np.asarray([np.asarray(p,dtype=float) for p,v in model.production_rate_map])
rates=np.asarray([float(v[idx]) for p,v in model.production_rate_map])
if np.any(rates<0):
    # Preserve signed rate information rather than disguising reversibility.
    scale=max(np.percentile(np.abs(rates),75),1e-30)
    color=np.arcsinh(rates/scale)
    clabel="asinh(net epoxide production rate / scale)"
else:
    color=np.log10(np.maximum(rates,1e-40))
    clabel="log10(epoxide production rate, model units)"
fig,ax=plt.subplots(figsize=(8.4,6.2))
cs=ax.tricontourf(pts[:,0],pts[:,1],color,levels=22)
fig.colorbar(cs,ax=ax,label=clabel)
with Path("energies.txt").open(encoding="utf-8",newline="") as fh:
    originals=list(csv.DictReader(fh,delimiter="\t"))
pairs={}
for row in originals:
    if row["surface_name"] not in ("None",):
        pairs.setdefault(row["surface_name"],{})[row["species_name"]]=float(row["formation_energy"])
for surface in model.surface_names:
    x,y=pairs[surface]["O"],pairs[surface]["Ha"]
    ax.scatter(x,y,s=28,c="white",edgecolors="black")
    ax.annotate(surface,(x,y),xytext=(4,3),textcoords="offset points",fontsize=8)
ax.set_xlabel("O* formation energy (eV)")
ax.set_ylabel("Ha* formation energy (eV)")
ax.set_title("C/O/H-selected 2D CatMAP activity volcano (8-step model)")
fig.tight_layout()
try:
    fig.savefig("production_rate_volcano.pdf")
    print("Saved production_rate_volcano.pdf")
except PermissionError:
    backup_dir.mkdir(parents=True,exist_ok=True)
    fig.savefig(backup_dir/"production_rate_volcano.pdf")
    print("Saved",backup_dir/"production_rate_volcano.pdf")
plt.close(fig)


In [ ]:
# Original tabular data export workflow, now including the selected descriptors.
import csv
variable="production_rate"
labels=[str(x) for x in model.output_labels[variable]]
with Path("production_rate_table.txt").open("w",encoding="utf-8",newline="") as fh:
    writer=csv.writer(fh,delimiter="\t")
    writer.writerow(["descriptor-"+d for d in model.descriptor_names]+labels)
    for pt,vals in getattr(model,variable+"_map"):
        writer.writerow([float(x) for x in pt]+[float(x) for x in vals])
print("Wrote production_rate_table.txt")


In [ ]:
# Retained original surface-coverage inspection logic.
labels=model.output_labels["coverage"]
for pt,cv in model.coverage_map[:5]:
    print("descriptors:",pt,"species:",labels,"coverages:",[float(v) for v in cv])


In [ ]:
# Optional: original rate-control, reaction-order, and selectivity analyses.
# Enable only when you need the expensive additional solver maps:
# for key in ["rate_control", "selectivity_control", "rxn_order"]:
#     if key not in model.output_variables:
#         model.output_variables.append(key)
# model.run()
